In [7]:
# Group_Integration_Pipeline.ipynb
# Group: MOVIE-SENTIBOT (IT24100209, IT24100095, IT24100163, IT24100068, IT24100109, IT24100156)
# Combined Preprocessing Pipeline for IT2011 Progress Review I
# Justification: Integrates all steps to demonstrate collaboration and logical flow for a complete preprocessing pipeline.
# Technique: Sequential application of all preprocessing steps.

# Import libraries
import pandas as pd
import re
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns
import os

%matplotlib inline

# Create output folder
os.makedirs('../results/outputs', exist_ok=True)

# Step 0: Load original dataset
df = pd.read_csv('../data/raw/archive/IMDB_Dataset.csv')
print("Original shape:", df.shape)

# Group: Missing Data Handling
missing = df.isnull().sum()
print("Missing values:\n", missing)
df = df.dropna()
print("Shape after missing handling:", df.shape)
df.to_csv('../results/outputs/missing_handled.csv', index=False)

# IT24100209: Data Cleaning
df = pd.read_csv('../results/outputs/missing_handled.csv')
df['word_count_before'] = df['review'].apply(lambda x: len(x.split()))
def clean_text(text): return re.sub('<.*?>', '', re.sub(r'[^\w\s]', '', text.lower()))
df['review'] = df['review'].apply(clean_text)
df['word_count_after'] = df['review'].apply(lambda x: len(x.split()))
df = df.drop(['word_count_before', 'word_count_after'], axis=1)
df.to_csv('../results/outputs/cleaned.csv', index=False)

# IT24100095: Encoding
df = pd.read_csv('../results/outputs/cleaned.csv')
le = LabelEncoder()
df['sentiment_encoded'] = le.fit_transform(df['sentiment'])
df.to_csv('../results/outputs/encoded.csv', index=False)

# IT24100163: Outlier Removal
df = pd.read_csv('../results/outputs/encoded.csv')
df['review_length'] = df['review'].apply(len)
Q1, Q3 = df['review_length'].quantile([0.25, 0.75])
IQR = Q3 - Q1
df = df[(df['review_length'] >= Q1 - 1.5 * IQR) & (df['review_length'] <= Q3 + 1.5 * IQR)]
df = df.drop('review_length', axis=1)
df.to_csv('../results/outputs/outliers_removed.csv', index=False)

# IT24100068: Normalization/Scaling
df = pd.read_csv('../results/outputs/outliers_removed.csv')
vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
X_tfidf = vectorizer.fit_transform(df['review']).toarray()
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_tfidf)
df_tfidf = pd.DataFrame(X_scaled, columns=[f'feature_{i}' for i in range(X_scaled.shape[1])])
df_tfidf['sentiment_encoded'] = df['sentiment_encoded'].values
df_tfidf.to_csv('../results/outputs/normalized.csv', index=False)

# IT24100109: Feature Selection
df = pd.read_csv('../results/outputs/normalized.csv')
X = df.drop('sentiment_encoded', axis=1).values
y = df['sentiment_encoded'].values
X_tfidf = vectorizer.fit_transform(pd.read_csv('../results/outputs/outliers_removed.csv')['review']).toarray()
selector = SelectKBest(chi2, k=10)
X_selected = selector.fit_transform(X_tfidf, y)
selected_features = [vectorizer.get_feature_names_out()[i] for i in selector.get_support(indices=True)]
df_selected = pd.DataFrame(X_selected, columns=selected_features)
df_selected['sentiment_encoded'] = y
df_selected.to_csv('../results/outputs/features_selected.csv', index=False)

# IT24100156: Dimension Reduction
df = pd.read_csv('../results/outputs/features_selected.csv')
X = df.drop('sentiment_encoded', axis=1).values
y = df['sentiment_encoded'].values
pca = PCA(n_components=2)
X_reduced = pca.fit_transform(X)
df_reduced = pd.DataFrame(X_reduced, columns=['PC1', 'PC2'])
df_reduced['sentiment_encoded'] = y
df_reduced.to_csv('../results/outputs/dimension_reduced.csv', index=False)

print("Pipeline completed. Final data saved to: processed_data/dimension_reduced.csv")

Original shape: (50000, 2)
Missing values:
 review       0
sentiment    0
dtype: int64
Shape after missing handling: (50000, 2)


MemoryError: Unable to allocate 353. MiB for an array with shape (46243, 1000) and data type float64